# Phase 5 prime bundle-first n=10 controls (Colab)

This notebook runs the existing `experiments/44_phase5_prime_bundle_first.py` harness on Colab. It is a Phase 5 prime diagnostic, not a Phase 5 graduation run.

The default path avoids Google Drive mount and writes results under `/content`. Download result JSONs from the final cell if you want to preserve them.

Planned sequence:

1. Clone `Dypatterson/Neuro-AI` and check out `phase5-m1-role-energy-stack`.
2. Run a tiny smoke.
3. Run the hard-cell n=10 control rerun in parallel shards: `D=4096`, `N=512`, `K_roles=16`, `cue_noise=0.15`, `scene_token=1`, skewed co-occurrence.
4. Run a candidate-only n=10 grid in parallel shards across K, N, cue noise, scene-token, and co-occurrence.
5. Run the scene-token follow-up: weight sweep plus reused-token controls on the hard skewed cells.
6. Run a context-bundle-anchor follow-up to test a full substrate-derived scene/context trace.
7. Run stricter partial-context follow-ups that exclude the queried role or use an observed prefix.
8. Run a targeted K=16 observed-prefix context-size curve before any full matrix.
9. Run a fixed observed-prefix operating-point grid across K, N, and noise.
10. Run a stricter available-prefix context-source follow-up where the observed prefix roles are sampled in the query plan.
11. Keep the full all-controls matrix as an explicit opt-in parallel-sharded cell because it is much larger.

Default parallelism is `MAX_PARALLEL = 24`. The harness now batches queries inside each shard, so GPU compute should be much higher than the original serial-query notebook. Lower parallelism if Colab becomes unstable.


In [ ]:
# 1. Clone the repo and check out the active branch.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git /content/Neuro-AI
%cd /content/Neuro-AI
!git checkout phase5-m1-role-energy-stack
!git log --oneline -1

# Verify the Phase 5 prime harness exists on this branch.
!test -f experiments/44_phase5_prime_bundle_first.py
!grep -n "bundle-first structural-memory" experiments/44_phase5_prime_bundle_first.py | head -1

In [ ]:
# 2. Runtime, harness sanity, and parallel shard helpers.
import json, os, subprocess, time
from pathlib import Path

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
print("device", DEVICE)

if DEVICE == "cuda":
    !nvidia-smi --query-gpu=name,memory.total --format=csv

!PYTHONPATH=src:. python -m py_compile experiments/44_phase5_prime_bundle_first.py

MAX_PARALLEL = 24  # Lower if Colab becomes unstable; raise only if GPU is still underused.
HARNESS = "experiments/44_phase5_prime_bundle_first.py"
ALL_CONDITIONS = [
    "candidate",
    "random_role",
    "shuffled_role",
    "deranged_role",
    "perfect_cue",
    "bundle_positive",
    "content_cleanup_positive",
]

def merge_payloads(part_paths, merged_out, label):
    payloads = [json.loads(Path(p).read_text()) for p in part_paths]
    merged = {
        "framing": payloads[0].get("framing", {}),
        "config": {
            "label": label,
            "parallel_shards": [str(p) for p in part_paths],
            "merged_by_notebook": True,
        },
        "aggregates": {},
        "raw": [],
    }
    for payload in payloads:
        for key, value in payload.get("aggregates", {}).items():
            if key in merged["aggregates"]:
                raise ValueError(f"duplicate aggregate key while merging: {key}")
            merged["aggregates"][key] = value
        merged["raw"].extend(payload.get("raw", []))
    Path(merged_out).write_text(json.dumps(merged, indent=2))
    print(f"merged {len(part_paths)} shards -> {merged_out}")
    return merged

def _launch_shard(name, args, out_root):
    out_path = out_root / f"{name}.json"
    log_path = out_root / f"{name}.log"
    cmd = ["python", HARNESS, *map(str, args), "--out", str(out_path)]
    env = dict(os.environ, PYTHONPATH="src:.")
    logf = open(log_path, "w")
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env)
    return {"name": name, "proc": proc, "logf": logf, "out": out_path, "log": log_path, "cmd": cmd}

def run_parallel_shards(label, shards, merged_out, max_parallel=MAX_PARALLEL, poll_s=20):
    out_root = Path(merged_out).with_suffix("")
    out_root.mkdir(parents=True, exist_ok=True)
    pending = list(shards)
    running = []
    completed = []
    failed = []
    t0 = time.time()
    print(f"{label}: {len(pending)} shards, max_parallel={max_parallel}, device={DEVICE}")
    while pending or running:
        while pending and len(running) < max_parallel:
            name, args = pending.pop(0)
            shard = _launch_shard(name, args, out_root)
            running.append(shard)
            print(f"  launched {name}")
        time.sleep(poll_s)
        still = []
        for shard in running:
            code = shard["proc"].poll()
            if code is None:
                still.append(shard)
                continue
            shard["logf"].close()
            elapsed = (time.time() - t0) / 60.0
            if code == 0 and shard["out"].exists():
                completed.append(shard)
                print(f"  done {shard['name']} at {elapsed:.1f} min")
            else:
                failed.append((shard, code))
                print(f"  FAILED {shard['name']} code={code} at {elapsed:.1f} min")
        running = still
        if DEVICE == "cuda":
            subprocess.run(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu", "--format=csv,noheader"], check=False)
    if failed:
        for shard, code in failed:
            print(f"\n--- tail {shard['log']} ---")
            lines = Path(shard["log"]).read_text(errors="replace").splitlines()
            print("\n".join(lines[-80:]))
        raise RuntimeError(f"{label}: {len(failed)} shard(s) failed")
    return merge_payloads([s["out"] for s in completed], merged_out, label)


In [ ]:
# 3. Tiny smoke. Expected: candidate > controls, positives at 1.0 on this small cell.
!PYTHONPATH=src:. python experiments/44_phase5_prime_bundle_first.py \
  --Ds 128 --Ns 8 --K_roles 2 --cue_noise 0.0 \
  --seeds 17 --n_queries 8 --C_codebook 32 \
  --conditions candidate random_role shuffled_role deranged_role perfect_cue bundle_positive content_cleanup_positive \
  --scene_token 0 --cooccurrence uniform --device cpu \
  --out /content/phase5_prime_smoke.json

In [ ]:
# 4. Hard-cell n=10 controls, sharded by condition.
# This reproduces the most important Report 069 cell while running controls concurrently.
HARD_CELL_OUT = "/content/phase5_prime_bundle_hard_cell_n10.json"
HARD_CONDITIONS = ALL_CONDITIONS
hard_shards = []
for condition in HARD_CONDITIONS:
    args = [
        "--Ds", 4096,
        "--Ns", 512,
        "--K_roles", 16,
        "--cue_noise", 0.15,
        "--seeds", 17, 11, 23, 1, 2, 3, 5, 7, 13, 29,
        "--n_queries", 512,
        "--C_codebook", 1024,
        "--conditions", condition,
        "--scene_token", 1,
        "--cooccurrence", "skewed",
        "--device", DEVICE,
    ]
    hard_shards.append((f"hard_{condition}", args))

hard_payload = run_parallel_shards("hard-cell-controls", hard_shards, HARD_CELL_OUT)

In [ ]:
# 5. Summarize the hard-cell controls.
def summarize_payload(path):
    payload = json.loads(Path(path).read_text())
    rows = []
    for key, agg in payload["aggregates"].items():
        rows.append((key, agg))
    for key, agg in sorted(rows):
        print(
            f"{key}\n"
            f"  top1={agg['top1_mean']:.4f} CI=[{agg['wilson_lo']:.4f},{agg['wilson_hi']:.4f}] "
            f"scene_tix={agg['scene_tix_rate']:.4f} content_tix={agg['content_tix_rate']:.4f} "
            f"n={agg['n_total']}"
        )
    return payload

hard_payload = summarize_payload(HARD_CELL_OUT)

In [ ]:
# 6. Candidate-only n=10 grid, sharded by K x scene-token x co-occurrence.
# This launches 16 independent shards and keeps up to MAX_PARALLEL active at once.
CANDIDATE_GRID_OUT = "/content/phase5_prime_candidate_grid_n10.json"
candidate_shards = []
for K in [2, 4, 8, 16]:
    for scene_token in [0, 1]:
        for cooc in ["uniform", "skewed"]:
            args = [
                "--Ds", 4096,
                "--Ns", 16, 32, 64, 128, 256, 512,
                "--K_roles", K,
                "--cue_noise", 0.0, 0.05, 0.10, 0.15,
                "--seeds", 17, 11, 23, 1, 2, 3, 5, 7, 13, 29,
                "--n_queries", 256,
                "--C_codebook", 1024,
                "--conditions", "candidate",
                "--scene_token", scene_token,
                "--cooccurrence", cooc,
                "--device", DEVICE,
            ]
            candidate_shards.append((f"candidate_K{K}_scene{scene_token}_{cooc}", args))

candidate_payload = run_parallel_shards("candidate-grid", candidate_shards, CANDIDATE_GRID_OUT)

In [ ]:
# 7. Summarize candidate grid: worst and best cells by top1.
candidate_payload = json.loads(Path(CANDIDATE_GRID_OUT).read_text())
rows = sorted(
    ((agg["top1_mean"], key, agg) for key, agg in candidate_payload["aggregates"].items()),
    key=lambda x: x[0],
)

print("Worst 20 candidate cells:")
for top1, key, agg in rows[:20]:
    print(f"{top1:.4f} CI=[{agg['wilson_lo']:.4f},{agg['wilson_hi']:.4f}] scene={agg['scene_tix_rate']:.4f} content={agg['content_tix_rate']:.4f} :: {key}")

print("\nBest 20 candidate cells:")
for top1, key, agg in rows[-20:]:
    print(f"{top1:.4f} CI=[{agg['wilson_lo']:.4f},{agg['wilson_hi']:.4f}] scene={agg['scene_tix_rate']:.4f} content={agg['content_tix_rate']:.4f} :: {key}")

In [ ]:
# 8. Scene-token follow-up: hard skewed cells with token-weight and token-reuse sweeps.
# token_pool=0 means unique scene token per scene; 1 means one shared/global token; 16 reuses anchors.
SCENE_TOKEN_FOLLOWUP_OUT = "/content/phase5_prime_scene_token_followup_n10.json"
FOLLOWUP_CONDITIONS = ["candidate", "random_role", "shuffled_role", "deranged_role", "content_cleanup_positive"]
followup_shards = []
for condition in FOLLOWUP_CONDITIONS:
    for K in [4, 8, 16]:
        for token_pool in [0, 1, 16]:
            args = [
                "--Ds", 4096,
                "--Ns", 512,
                "--K_roles", K,
                "--cue_noise", 0.15,
                "--seeds", 17, 11, 23, 1, 2, 3, 5, 7, 13, 29,
                "--n_queries", 512,
                "--C_codebook", 1024,
                "--conditions", condition,
                "--scene_token", 1,
                "--scene_token_weight", 0.0, 0.1, 0.25, 0.5, 1.0,
                "--scene_token_source", "random",
                "--scene_token_pool_size", token_pool,
                "--cooccurrence", "skewed",
                "--device", DEVICE,
            ]
            followup_shards.append((f"scene_token_{condition}_K{K}_pool{token_pool}", args))

scene_token_followup_payload = run_parallel_shards(
    "scene-token-followup",
    followup_shards,
    SCENE_TOKEN_FOLLOWUP_OUT,
)


In [ ]:
# 9. Summarize scene-token follow-up rows.
scene_token_followup_payload = json.loads(Path(SCENE_TOKEN_FOLLOWUP_OUT).read_text())
followup_rows = sorted(scene_token_followup_payload["aggregates"].items())
for key, agg in followup_rows:
    print(
        f"{agg['top1_mean']:.4f} CI=[{agg['wilson_lo']:.4f},{agg['wilson_hi']:.4f}] "
        f"scene={agg['scene_tix_rate']:.4f} content={agg['content_tix_rate']:.4f} :: {key}"
    )


In [ ]:
# 10. Context-bundle anchor follow-up.
# This replaces random scene IDs with a substrate-derived context trace: the role-filler scene bundle itself.
CONTEXT_ANCHOR_OUT = "/content/phase5_prime_context_anchor_followup_n10.json"
context_shards = []
for condition in FOLLOWUP_CONDITIONS:
    for K in [4, 8, 16]:
        args = [
            "--Ds", 4096,
            "--Ns", 512,
            "--K_roles", K,
            "--cue_noise", 0.15,
            "--seeds", 17, 11, 23, 1, 2, 3, 5, 7, 13, 29,
            "--n_queries", 512,
            "--C_codebook", 1024,
            "--conditions", condition,
            "--scene_token", 1,
            "--scene_token_weight", 0.1, 0.25, 0.5, 1.0,
            "--scene_token_source", "context_bundle",
            "--scene_token_pool_size", 0,
            "--cooccurrence", "skewed",
            "--device", DEVICE,
        ]
        context_shards.append((f"context_anchor_{condition}_K{K}", args))

context_anchor_payload = run_parallel_shards(
    "context-anchor-followup",
    context_shards,
    CONTEXT_ANCHOR_OUT,
)


In [ ]:
# 11. Summarize context-anchor follow-up rows.
context_anchor_payload = json.loads(Path(CONTEXT_ANCHOR_OUT).read_text())
context_rows = sorted(context_anchor_payload["aggregates"].items())
for key, agg in context_rows:
    print(
        f"{agg['top1_mean']:.4f} CI=[{agg['wilson_lo']:.4f},{agg['wilson_hi']:.4f}] "
        f"scene={agg['scene_tix_rate']:.4f} content={agg['content_tix_rate']:.4f} :: {key}"
    )


In [ ]:
# 12. Strict context-anchor variants.
# These keep storage fixed but restrict query-side context so the queried role/filler is not leaked.
STRICT_CONTEXT_ANCHOR_OUT = "/content/phase5_prime_strict_context_anchor_followup_n10.json"
STRICT_CONTEXT_SOURCES = [
    "context_bundle_exclude_query_role",
    "context_bundle_observed_prefix",
]
strict_context_shards = []
for condition in FOLLOWUP_CONDITIONS:
    for K in [4, 8, 16]:
        for source in STRICT_CONTEXT_SOURCES:
            args = [
                "--Ds", 4096,
                "--Ns", 512,
                "--K_roles", K,
                "--cue_noise", 0.15,
                "--seeds", 17, 11, 23, 1, 2, 3, 5, 7, 13, 29,
                "--n_queries", 512,
                "--C_codebook", 1024,
                "--conditions", condition,
                "--scene_token", 1,
                "--scene_token_weight", 0.1, 0.25, 0.5, 1.0,
                "--scene_token_source", source,
                "--scene_token_pool_size", 0,
                "--cooccurrence", "skewed",
                "--device", DEVICE,
            ]
            if source == "context_bundle_observed_prefix":
                args.extend(["--context_roles", 1, 2, min(4, K - 1)])
            strict_context_shards.append((f"strict_context_{condition}_K{K}_{source}", args))

strict_context_payload = run_parallel_shards(
    "strict-context-anchor-followup",
    strict_context_shards,
    STRICT_CONTEXT_ANCHOR_OUT,
)


In [ ]:
# 13. Summarize strict context-anchor follow-up rows.
strict_context_payload = summarize_payload(STRICT_CONTEXT_ANCHOR_OUT)


In [ ]:
# 14. K=16 observed-prefix context-size curve.
# This is the targeted follow-up after the strict-context pass: locate the context-size threshold.
OBSERVED_PREFIX_CURVE_OUT = "/content/phase5_prime_observed_prefix_curve_k16_n10.json"
observed_prefix_curve_shards = []
for condition in FOLLOWUP_CONDITIONS:
    args = [
        "--Ds", 4096,
        "--Ns", 512,
        "--K_roles", 16,
        "--cue_noise", 0.15,
        "--seeds", 17, 11, 23, 1, 2, 3, 5, 7, 13, 29,
        "--n_queries", 512,
        "--C_codebook", 1024,
        "--conditions", condition,
        "--scene_token", 1,
        "--scene_token_weight", 0.1, 0.25, 0.5,
        "--scene_token_source", "context_bundle_observed_prefix",
        "--scene_token_pool_size", 0,
        "--context_roles", 1, 2, 3, 4, 6, 8,
        "--cooccurrence", "skewed",
        "--device", DEVICE,
    ]
    observed_prefix_curve_shards.append((f"observed_prefix_curve_K16_{condition}", args))

observed_prefix_curve_payload = run_parallel_shards(
    "observed-prefix-curve-k16",
    observed_prefix_curve_shards,
    OBSERVED_PREFIX_CURVE_OUT,
)


In [ ]:
# 15. Summarize K=16 observed-prefix context-size curve.
observed_prefix_curve_payload = summarize_payload(OBSERVED_PREFIX_CURVE_OUT)


In [ ]:
# 16. Fixed observed-prefix operating-point grid.
# Selected from the K=16 context-size curve: context_roles=4, token_weight=0.25.
FIXED_OBSERVED_PREFIX_OUT = "/content/phase5_prime_fixed_observed_prefix_grid_n10.json"
fixed_observed_prefix_shards = []
for condition in FOLLOWUP_CONDITIONS:
    for K in [4, 8, 16]:
        args = [
            "--Ds", 4096,
            "--Ns", 128, 256, 512,
            "--K_roles", K,
            "--cue_noise", 0.0, 0.10, 0.15,
            "--seeds", 17, 11, 23, 1, 2, 3, 5, 7, 13, 29,
            "--n_queries", 512,
            "--C_codebook", 1024,
            "--conditions", condition,
            "--scene_token", 1,
            "--scene_token_weight", 0.25,
            "--scene_token_source", "context_bundle_observed_prefix",
            "--scene_token_pool_size", 0,
            "--context_roles", 4,
            "--cooccurrence", "skewed",
            "--device", DEVICE,
        ]
        fixed_observed_prefix_shards.append((f"fixed_observed_prefix_{condition}_K{K}", args))

fixed_observed_prefix_payload = run_parallel_shards(
    "fixed-observed-prefix-grid",
    fixed_observed_prefix_shards,
    FIXED_OBSERVED_PREFIX_OUT,
)


In [ ]:
# 17. Summarize fixed observed-prefix operating-point grid.
fixed_observed_prefix_payload = summarize_payload(FIXED_OBSERVED_PREFIX_OUT)


In [ ]:
# 18. Available-prefix context-source follow-up.
# This stricter source samples the observed non-query prefix roles in the query plan
# instead of filling hidden non-query roles deterministically by index.
AVAILABLE_PREFIX_CONTEXT_OUT = "/content/phase5_prime_available_prefix_context_n10.json"
available_prefix_context_shards = []
for condition in FOLLOWUP_CONDITIONS:
    args = [
        "--Ds", 4096,
        "--Ns", 512,
        "--K_roles", 16,
        "--cue_noise", 0.15,
        "--seeds", 17, 11, 23, 1, 2, 3, 5, 7, 13, 29,
        "--n_queries", 512,
        "--C_codebook", 1024,
        "--conditions", condition,
        "--scene_token", 1,
        "--scene_token_weight", 0.1, 0.25, 0.5,
        "--scene_token_source", "context_bundle_observed_prefix_plan",
        "--scene_token_pool_size", 0,
        "--context_roles", 1, 2, 3, 4, 6, 8,
        "--cooccurrence", "skewed",
        "--device", DEVICE,
    ]
    available_prefix_context_shards.append((f"available_prefix_context_K16_{condition}", args))
available_prefix_context_payload = run_parallel_shards(
    "available-prefix-context-k16",
    available_prefix_context_shards,
    AVAILABLE_PREFIX_CONTEXT_OUT,
)


In [ ]:
# 19. Summarize available-prefix context-source follow-up.
available_prefix_context_payload = summarize_payload(AVAILABLE_PREFIX_CONTEXT_OUT)


In [ ]:
# 20. Optional full all-controls matrix, sharded by condition x K x scene-token x co-occurrence.
# This creates 112 shards. Keep MAX_PARALLEL conservative if Colab becomes unstable.
RUN_FULL_MATRIX = False

if RUN_FULL_MATRIX:
    FULL_MATRIX_OUT = "/content/phase5_prime_full_controls_grid_n10.json"
    full_shards = []
    for condition in ALL_CONDITIONS:
        for K in [2, 4, 8, 16]:
            for scene_token in [0, 1]:
                for cooc in ["uniform", "skewed"]:
                    args = [
                        "--Ds", 4096,
                        "--Ns", 16, 32, 64, 128, 256, 512,
                        "--K_roles", K,
                        "--cue_noise", 0.0, 0.05, 0.10, 0.15,
                        "--seeds", 17, 11, 23, 1, 2, 3, 5, 7, 13, 29,
                        "--n_queries", 512,
                        "--C_codebook", 1024,
                        "--conditions", condition,
                        "--scene_token", scene_token,
                        "--cooccurrence", cooc,
                        "--device", DEVICE,
                    ]
                    full_shards.append((f"full_{condition}_K{K}_scene{scene_token}_{cooc}", args))
    full_payload = run_parallel_shards("full-controls-grid", full_shards, FULL_MATRIX_OUT)
else:
    print("Full all-controls matrix skipped. Set RUN_FULL_MATRIX = True to run it.")

In [ ]:
# 21. Download result JSONs from the Colab runtime.
from google.colab import files

for path in [
    "/content/phase5_prime_smoke.json",
    "/content/phase5_prime_bundle_hard_cell_n10.json",
    "/content/phase5_prime_candidate_grid_n10.json",
    "/content/phase5_prime_scene_token_followup_n10.json",
    "/content/phase5_prime_context_anchor_followup_n10.json",
    "/content/phase5_prime_strict_context_anchor_followup_n10.json",
    "/content/phase5_prime_observed_prefix_curve_k16_n10.json",
    "/content/phase5_prime_fixed_observed_prefix_grid_n10.json",
    "/content/phase5_prime_available_prefix_context_n10.json",
    "/content/phase5_prime_full_controls_grid_n10.json",
]:
    if Path(path).exists():
        print("downloading", path)
        files.download(path)
